# ARC2 011 CPU pilot preflight

Local preparation only; publication and execution are separate. 2026-10-11. Fixed two development tasks; no confirmation tasks or competition submission. Base initialization is provisional pending the operator's S0 decision. CPU diagnostics load tokenizer assets and an optional tiny random interface model, never 27B weights.


### 1. Restore independently hashed core code and P2-only inputs


In [ ]:
import base64, hashlib, io, json, pathlib, time, zipfile
NOTEBOOK_START=time.monotonic()
CODE=pathlib.Path('/kaggle/working/011/pilot-code'); CODE.mkdir(parents=True,exist_ok=True)
CODE_MANIFEST={'arc_episode_data.py': '7c672182a1bff3654231f9e60da2ba503ae94124b215d04857cb20345d93c864', 'grid_runtime.py': '54b519deb5876f0f9f90008da69151b6ec4aa35b808b7c80d7d624f385c7d20f', 'p2-challenges.json': 'abef4b67006970e7a899e9a15f661d65ec56b6f3f4264bf5c667717f916804d3', 'p2-queries.json': '6379a22f3581b32411b7547f187c57403dbd6ba07fa22875d3b6e4a63dbd8b61', 'p2-scoring.json': '2efbc9bf7437fd332521571aa582fc2126b30b9ff8658280a8d31c331ed08daf', 'pilot-info.json': 'ec6cea27c88abe4808b94c751c231e6b69b699fcf44dbd8d844d821833f7842b', 'run_budgeted_grid_experiment.py': 'af5ae4f2baca3f1c670f9b50fc94b98dc46c04eaeea1cf55ec34ea0497107e23', 'solve_grid.py': 'c29c428d87873c32be29fd394b8d06cdcee37da694723d49a677d25f699ca190', 'solve_program.py': 'bb500aef69a90b9613f74e0e33b44895f7d3b9771138481b500ba836605bb54d', 'train_grid_sft.py': '2e4ccb40b7acee2c3b5659eff9a6afa4bcd57fa65ee78f5a1ee6e1dda285f044', 'ttt_per_task.py': '17d32077cc788457ae6943de6c4944453c280d4ef515220915920321a42d7229'}
CORE_MANIFEST={'arc_episode_data.py': '7c672182a1bff3654231f9e60da2ba503ae94124b215d04857cb20345d93c864', 'grid_runtime.py': '54b519deb5876f0f9f90008da69151b6ec4aa35b808b7c80d7d624f385c7d20f', 'run_budgeted_grid_experiment.py': 'af5ae4f2baca3f1c670f9b50fc94b98dc46c04eaeea1cf55ec34ea0497107e23', 'solve_grid.py': 'c29c428d87873c32be29fd394b8d06cdcee37da694723d49a677d25f699ca190', 'solve_program.py': 'bb500aef69a90b9613f74e0e33b44895f7d3b9771138481b500ba836605bb54d', 'train_grid_sft.py': '2e4ccb40b7acee2c3b5659eff9a6afa4bcd57fa65ee78f5a1ee6e1dda285f044', 'ttt_per_task.py': '17d32077cc788457ae6943de6c4944453c280d4ef515220915920321a42d7229'}
CORE_MANIFEST_SHA='bbc9b83fc24f7b216878695cf7ca41fec83af15ab94bbb8af640047e2896a97e'
PILOT_TASKS=('db118e2a', '291dc1e1')
EXPECTED_ADAPTER_SHA='9282c46cbce02923afbea3ad041614f0bc6ff3d7bd97c5174adfa3dd3e485ff9'
payload=base64.b64decode('...')
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert set(z.namelist())==set(CODE_MANIFEST)
    for name in z.namelist():
        if pathlib.PurePosixPath(name).name!=name: raise ValueError('unsafe archive path')
        data=z.read(name)
        assert hashlib.sha256(data).hexdigest()==CODE_MANIFEST[name]
        (CODE/name).write_bytes(data)
(CODE/'source-manifest.json').write_text(json.dumps(CODE_MANIFEST,indent=2))
(CODE/'core-manifest.json').write_text(json.dumps(CORE_MANIFEST,indent=2))
print('PILOT_CORE_SHA',CORE_MANIFEST_SHA)


### 2. Prepare the frozen offline dependencies


In [ ]:
import glob, os, subprocess, sys, time, json
wheel = glob.glob("/kaggle/input/**/qwen38-wheelhouse-py312", recursive=True)
if len(wheel) != 1:
    raise RuntimeError("Expected one frozen wheelhouse, found %d" % len(wheel))
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
               stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, timeout=90)
p = subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                    "--find-links=" + wheel[0], "transformers==5.8.0", "accelerate", "peft"],
                   capture_output=True, text=True, timeout=240)
print((p.stdout or "")[-1400:])
if p.returncode:
    print((p.stderr or "")[-1400:])
    raise RuntimeError("Offline environment preparation failed")
import transformers, torch, peft, accelerate
versions = dict(python=sys.version.split()[0], transformers=transformers.__version__,
                torch=torch.__version__, peft=peft.__version__, accelerate=accelerate.__version__)
print("VERSIONS", json.dumps(versions))
assert transformers.__version__ == "5.8.0"


### 3. Verify model and mounted S4 adapter identities

The S4 adapter is verified but neither pilot uses it.


In [ ]:
from pathlib import Path
import hashlib
def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(2*1024*1024), b""):
            h.update(b)
    return h.hexdigest()
models = []
for p in Path("/kaggle/input").rglob("config.json"):
    try:
        c = json.loads(p.read_text())
    except Exception:
        continue
    if c.get("model_type") == "qwen3_5" and (p.parent/"model.safetensors.index.json").is_file():
        models.append(p.parent)
if len(models) != 1:
    raise RuntimeError("Expected one Qwen3.8 model directory, found %d" % len(models))
model_dir = str(models[0])
adapters = list(Path("/kaggle/input").rglob("adapter_step_25/adapter_model.safetensors"))
matching = [p.parent for p in adapters if digest(p) == EXPECTED_ADAPTER_SHA]
if len(matching) != 1:
    raise RuntimeError("Expected verified S4 adapter SHA, matching directories=%d" % len(matching))
adapter_dir = str(matching[0])
assert json.loads((matching[0]/"formatter.json").read_text())["formatter"] == "f0_space"
print("INPUT_IDENTITY", json.dumps(dict(model_dir=model_dir, adapter_dir=adapter_dir,
     adapter_sha256=EXPECTED_ADAPTER_SHA, model_config_sha256=digest(models[0]/"config.json"))))
sys.path.insert(0, str(CODE))


### 4. Validate the bounded P2 configuration

TTT: A0/A4, O+T, 4 updates × 4 accumulation. Its 3600-second stage limit includes initial setup/loading. Program: shared base, initial/restart/repair, 1536 tokens / 360 seconds per call / 1080 seconds per task, stage limit 2400 seconds. The complete Notebook remains bounded by 7200 seconds, with 300 seconds reserved for saving.


In [ ]:
config = {
 "model_dir": model_dir, "challenges": str(CODE/"p2-challenges.json"),
 "solutions": str(CODE/"p2-scoring.json"), "out_dir": "/kaggle/working/011/s1-pilots",
 "time_budget_s": 7200, "load_limit_s": 1200,
 "notebook_start_monotonic": NOTEBOOK_START,
 "runtime": {"base_source":"kaggle:ugvfpdcuwfnh/qwen3-8-27b/Transformers/transformers-bf16/1",
             "deactivate_async_load":True, "save_window_s":300},
 "stages": [
  {"type":"ttt","name":"s1_base_f1_ttt_p2","adapter":None,"formatter":"f1_compact",
   "ttt_adapter_name":"ttt_p2_base","query_file":str(CODE/"p2-queries.json"),
   "updates":4,"max_new_tokens":960,"seed":20261011,"time_budget_s":3600,
   "a4_adaptation_estimate_s":480,"a4_generation_reserve_s_per_query":480,
   "source_by_task":{"db118e2a":"H_new","291dc1e1":"E24"}},
  {"type":"program","name":"s2_base_program_p2","adapter":None,
   "query_file":str(CODE/"p2-queries.json"),"time_budget_s":2400,"pilot_tasks":2,
   "limits":{"max_new_tokens":1536,"max_prompt_tokens":8192,
             "max_call_seconds":360,"max_task_seconds":1080,
             "max_feedback_chars":3000,"per_grid_seconds":2,
             "per_program_seconds":20,"memory_mb":512,"seed":20261011}}
 ]}
config_path=CODE/"run.json"
config_path.write_text(json.dumps(config,ensure_ascii=False,indent=2))
p=subprocess.run([sys.executable,str(CODE/"run_budgeted_grid_experiment.py"),
                  "--config",str(config_path),"--dry-run"],capture_output=True,text=True,timeout=60)
print(p.stdout)
if p.returncode:
    print(p.stderr[-1500:])
    raise RuntimeError("Pilot configuration preflight failed")


### 5. Linux CPU diagnostics, including the actual candidate orchestration path


In [ ]:
# Manual CPU diagnostics: real orchestration, synthetic generation, no 27B weights.
assert sys.platform.startswith("linux"),"Executor diagnostics require the Kaggle Linux image"
assert not torch.cuda.is_available(),"CPU preflight must not reserve a GPU"
from types import SimpleNamespace
from transformers import AutoTokenizer
import arc_episode_data, grid_runtime, run_budgeted_grid_experiment, solve_grid, solve_program, ttt_per_task, train_grid_sft
core_modules=[arc_episode_data,grid_runtime,run_budgeted_grid_experiment,solve_grid,
              solve_program,ttt_per_task,train_grid_sft]
for module in core_modules:
    path=Path(module.__file__).resolve()
    assert path.parent==CODE.resolve(),(module.__name__,str(path))
    assert digest(path)==CORE_MANIFEST[path.name],path.name
assert hasattr(run_budgeted_grid_experiment,"score_ttt_results")
assert hasattr(run_budgeted_grid_experiment,"stage_work_complete")
tasks=json.loads((CODE/"p2-challenges.json").read_text())
queries=json.loads((CODE/"p2-queries.json").read_text())
assert list(tasks)==list(PILOT_TASKS)
assert queries==[{"task_id":t,"query_index":0} for t in PILOT_TASKS]
assert all(set(q)=={"input"} for task in tasks.values() for q in task["test"])
assert all(stage.get("adapter") is None for stage in config["stages"])
tok=AutoTokenizer.from_pretrained(model_dir,local_files_only=True,trust_remote_code=False)
sample=tasks[PILOT_TASKS[0]]
messages=arc_episode_data.build_messages(sample["train"],sample["test"][0]["input"],"f1_compact")
off=tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
on=tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,
                           enable_thinking=True,reasoning_effort="low",preserve_thinking=True)
assert off!=on and on.endswith("<think>\n")
loo=[]
for tid in PILOT_TASKS:
    pool,detail=ttt_per_task.build_ttt_episodes(tok,tasks[tid]["train"],tid,
                    formatter="f1_compact",eos_id=tok.eos_token_id,max_len=4096,seed=20261011)
    assert detail["status"]=="complete",detail
    for episode in pool:
        assert all(x==-100 for x in episode.labels[:episode.prompt_len])
        assert all(x!=-100 for x in episode.labels[episode.prompt_len:])
        assert episode.labels[-1]==tok.eos_token_id
    loo.append(detail)

# Exercise the real generate_grid -> deadline wrapper -> _run_one_candidate ->
# decode/parse -> candidate journal path. Only FakeModel.generate is synthetic.
# This catches event-name/"kind" collisions without downloading/loading a model.
class FakeModel:
    def __init__(self):
        self.config=SimpleNamespace(use_cache=True)
        self.generation_config=SimpleNamespace(cache_implementation=None)
        self.calls=0
        self.answer="12\n34"
    def eval(self):
        return self
    def gradient_checkpointing_disable(self):
        pass
    def generate(self,input_ids,**kwargs):
        self.calls+=1
        assert "stopping_criteria" in kwargs and len(kwargs["stopping_criteria"])>=1
        for stop in kwargs["stopping_criteria"]:
            assert not bool(stop(input_ids,None))
        if kwargs.get("do_sample"):
            torch.multinomial(torch.tensor([0.5,0.5]),1)
        tail=tok.encode(self.answer,add_special_tokens=False)+[tok.eos_token_id]
        assert len(tail)<kwargs["max_new_tokens"]
        return torch.cat([input_ids,torch.tensor([tail],dtype=input_ids.dtype,device=input_ids.device)],dim=1)
fake=FakeModel()
fixture_session=grid_runtime.Session(model=fake,tokenizer=tok,torch=torch,
    in_dev=torch.device("cpu"),eos_id=tok.eos_token_id,pad_id=tok.pad_token_id,
    deadline=time.monotonic()+120,model_dir="synthetic-no-weights",
    base_identity={"kind":"cpu_fixture"},
    journal=grid_runtime.CandidateJournal("/kaggle/working/011/cpu-pilot-candidates.jsonl"),
    active_identity={"kind":"cpu_fixture"},save_window_s=0,allow_cached=True)
fixture_demos=[{"input":[[0]],"output":[[0]]}]
fixture_rows=[]
for fmt,text in [("f0_space","1 2\n3 4"),("f1_compact","12\n34")]:
    fake.answer=text
    for view,sampling in [("original",False),("original",True),("transpose",False)]:
        record=grid_runtime.generate_grid(fixture_session,"cpu_fixture_"+fmt,0,
           fixture_demos,[[1,2],[3,4]],formatter=fmt,view=view,do_sample=sampling,
           sample_index=1 if sampling else 0,max_new_tokens=32,seed=20261011,
           deadline=fixture_session.deadline,run_tag="cpu-pilot-orchestration")
        assert record["status"]=="complete",record
        assert record["grid"]==([[1,3],[2,4]] if view=="transpose" else [[1,2],[3,4]])
        fixture_rows.append({k:record.get(k) for k in ("identity_key","view","kind","status","parse","cache_hit")})
calls_before=fake.calls
cached=grid_runtime.generate_grid(fixture_session,"cpu_fixture_f1_compact",0,
    fixture_demos,[[1,2],[3,4]],formatter="f1_compact",view="transpose",do_sample=False,
    max_new_tokens=32,seed=20261011,deadline=fixture_session.deadline,run_tag="cpu-pilot-orchestration")
assert cached["cache_hit"] and fake.calls==calls_before
expired=grid_runtime.generate_grid(fixture_session,"cpu_fixture_expired",0,
    fixture_demos,[[0]],formatter="f1_compact",max_new_tokens=32,
    deadline=time.monotonic()-1,run_tag="cpu-pilot-expired")
assert expired["status"]=="budget_interrupted" and fake.calls==calls_before

# Tiny randomly initialized HF model: exercise the actual PEFT/generate API on
# CPU for two tokens. This is an interface probe, not a Qwen3.8 forward/ARC score.
from transformers import Qwen3Config,Qwen3ForCausalLM
tiny_config=Qwen3Config(vocab_size=64,hidden_size=32,intermediate_size=64,
    num_hidden_layers=1,num_attention_heads=4,num_key_value_heads=2,head_dim=8,
    max_position_embeddings=64,bos_token_id=1,eos_token_id=2,pad_token_id=0)
tiny=Qwen3ForCausalLM(tiny_config).eval()
tiny,initial=ttt_per_task.ensure_ttt_adapter(tiny,adapter_name="cpu_ttt_interface",
                                           seed=20261011,torch_module=torch)
with torch.inference_mode():
    tiny_result=tiny.generate(input_ids=torch.tensor([[1,4,5]]),
        attention_mask=torch.ones((1,3),dtype=torch.long),max_new_tokens=2,
        do_sample=False,pad_token_id=0)
assert 3<int(tiny_result.shape[1])<=5
assert all(not p.requires_grad for p in tiny.parameters())
tiny_probe={"status":"ok","model":"random_tiny_Qwen3_not_Qwen3_8_27B",
            "device":"cpu","max_new_tokens":2,"generated_tokens":int(tiny_result.shape[1])-3,
            "lora_r":8,"lora_alpha":32,"initial_adapter_sha256":initial["adapter_sha256"]}
del tiny_result,tiny,initial

executor=solve_program.executor_preflight()
assert executor["status"]=="ok",executor
cases=[
 ("identity","def transform(grid):\n    return [row[:] for row in grid]",2.0,"ok"),
 ("exception","def transform(grid):\n    return grid[100]",2.0,"execution_error"),
 ("timeout","def transform(grid):\n    while True:\n        pass",0.2,"execution_timeout"),
 ("import","import os\ndef transform(grid):\n    return grid",2.0,"program_rejected")]
case_results={}
for name,code,wall,want in cases:
    result=solve_program.execute_grid(code,[[0,1],[2,3]],wall_seconds=wall,memory_mb=512)
    case_results[name]=result
    assert result["status"]==want,(name,result)
assert case_results["identity"]["grid"]==[[0,1],[2,3]]
report={"status":"CPU_PILOT_PREFLIGHT_OK","model_weights_loaded":False,
        "pretrained_model_weights_loaded":False,"synthetic_cpu_model_instantiated":True,
        "qwen38_27b_forward_ran":False,"gpu_count":torch.cuda.device_count(),
        "versions":versions,"core_manifest":CORE_MANIFEST,"core_manifest_sha256":CORE_MANIFEST_SHA,
        "payload_manifest":CODE_MANIFEST,"adapter_sha256":EXPECTED_ADAPTER_SHA,
        "pilot_tasks":list(PILOT_TASKS),"ttt_loo":loo,
        "config_validated":True,"configuration_runtime_gpu_verified":False,
        "candidate_orchestration":{"status":"ok","model_generate":"synthetic_stub",
            "rows":fixture_rows,"generate_calls":fake.calls,"journal_cache_hit":True,
            "expired_admission":"budget_interrupted"},
        "tiny_peft_generate":tiny_probe,"executor":executor,"case_results":case_results,
        "thinking_prompt_tokens":len(tok.encode(on,add_special_tokens=False)),
        "nonthinking_prompt_tokens":len(tok.encode(off,add_special_tokens=False)),
        "elapsed_s":time.monotonic()-NOTEBOOK_START}
Path("/kaggle/working/011/cpu-pilot-preflight.json").write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(json.dumps(report,ensure_ascii=False,indent=2))


### Interpretation

A CPU pass is not a GPU gradient/throughput result. TTT here tests fixed-candidate adaptation, not equal-time superiority. Program repair/restart uses the same P0 and only demo feedback. Include incomplete work and actual costs; a parsed grid or low training loss is not a solved task. These are development diagnostics, not competition scores.
